# RFM + K-Means Customer Segmentation – Practice Skeleton

**Short name (GitHub):** `CustSeg`

**Lab source:** Online Retail RFM clustering (UCI-style extract) — clean line items, build Recency / Frequency / Monetary, log+scale, K-Means, PCA view.

Work this notebook first. Peek at `CustSeg_Solution.ipynb` only when stuck. `CustSeg.py` is the reusable helper module.

**Files**
- `data/online_retail.csv` — 36,110 line items (extract of the public Online Retail book, Dec 2010–Dec 2011)
- `custseg_flowchart.png` — desired outcome
- `CustSeg_Cheatsheet.docx`, `CustSeg_Project_Memo.docx`, `CustSeg_Strategy_Guide.docx`

**Not a loyalty program, not a next-purchase model, not a credit decision.** Clustering groups similar RFM rows. A human still names the bins and owns the treatment.



## Inline cheat-sheet (keep this cell visible)

See also **`CustSeg_Cheatsheet.docx`**.

| Item | Formula / code |
|------|----------------|
| Line item vs customer | row in `online_retail.csv` = one SKU on an invoice; RFM row = one `CustomerID` |
| TotalSum | \(Q \times P\) after `Quantity > 0` (and usually `UnitPrice > 0`) |
| Snapshot | \(\text{snapshot} = \max(\text{InvoiceDate}) + 1\text{ day}\) |
| Recency | \((\text{snapshot} - \max_i t_i).\text{days}\) — smaller is warmer |
| Frequency | `nunique(InvoiceNo)`, **not** line-item `count` |
| Monetary | \(\sum Q\cdot P\) over the window |
| Skew fix | `np.log1p` on R, F, M **before** scaling |
| Scale | \(x'=(x-\mu)/\sigma\) on the log frame; keep \(\mu,\sigma\) for new customers |
| Inertia | \(J=\sum_i\|x_i-c_{\ell_i}\|^2\) on the **scaled** matrix |
| Elbow / sil. | plot \(J(k)\); confirm with silhouette; this extract likes \(k=4\) |
| Name bins | groupby Cluster → **median** R/F/M in original units |
| PCA | a slide of the 3-D scaled space; PC1 ≈ value, PC2 ≈ recency |

**Order:** clean → RFM → log1p → scale → choose \(k\) → fit → profile originals → PCA last.



## Desired outcome

![flowchart](custseg_flowchart.png)

1. Load `data/online_retail.csv`. Drop missing `CustomerID`. Keep `Quantity > 0` (and `UnitPrice > 0` for a strict sales book).
2. `TotalSum = Quantity * UnitPrice`. Parse `InvoiceDate` with `%d.%m.%Y %H:%M`. Cast `CustomerID` to int.
3. `snapshot_date = max(InvoiceDate) + 1 day`. Aggregate Recency / Frequency / Monetary per customer.
4. `log1p` the three columns, then `StandardScaler`. Do not overwrite the original RFM table.
5. Elbow + silhouette for \(k=1\ldots10\). Fit K-Means at the chosen \(k\) (we use 4).
6. Write `rfm["Cluster"] = labels`. Profile **medians** in original units and name the bins.
7. PCA to 2-D is a picture, not the model. Alternates, more practice, then turn the simulation knobs.



## 0. Packages


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from sklearn.cluster import KMeans, MiniBatchKMeans, AgglomerativeClustering
    from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
    from sklearn.decomposition import PCA
    from sklearn.metrics import silhouette_score
    HAS_SK = True
except ImportError:
    HAS_SK = False
    print("sklearn not found — install scikit-learn to run the clustering cells.")

import CustSeg as cs

plt.rcParams["figure.figsize"] = (8, 4.5)
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", 20)
print("sklearn", HAS_SK, "| CustSeg helpers ready")


## 1. Why RFM then K-Means

Retail invoices are the wrong grain for a segment. One customer can place 30 SKUs on one ticket — that is **one visit**, not thirty. RFM collapses the book to one row per `CustomerID`:

- **Recency** — days since last paid invoice (relative to a snapshot).
- **Frequency** — distinct invoices, not line items.
- **Monetary** — lifetime spend in the window.

K-Means then partitions that 3-column table. It assumes roughly spherical blobs in Euclidean space, which is why we log-transform the long right tail and standardise before we call `.fit`.



## 2. Load and clean

This extract has ~25% missing `CustomerID` and a few hundred returns (`Quantity < 0`, invoices starting with `C`). Those rows cannot join a customer segment.

**Task 2.1** Read `data/online_retail.csv`. Print shape, dtypes, missing counts, and the `Quantity` / `UnitPrice` extremes.

**Task 2.2** Drop missing `CustomerID`. Keep `Quantity > 0` (zero-qty rows would inflate Frequency). Optionally drop `UnitPrice <= 0`. Build `TotalSum`. Parse dates with `format="%d.%m.%Y %H:%M"`. Cast `CustomerID` to int.

Sanity: no remaining negatives, no missing IDs, revenue and unique-customer counts printed.



In [ ]:
# Task 2.1 — inspect the raw extract
# raw = pd.read_csv("data/online_retail.csv")
# print shape / info / isna / Quantity describe / Country value_counts



In [ ]:
# Task 2.2 — clean into `sales`
# sales = ...
# sales["TotalSum"] = ...
# sales["InvoiceDate"] = pd.to_datetime(..., format="%d.%m.%Y %H:%M")
# print clean shape, unique customers, total revenue, min/max date



## 3. Look at the book before you cluster

**Task 3.1** Top countries by revenue. What share is the United Kingdom?

**Task 3.2** Histogram of line-item `TotalSum` (clip the axis — a few wholesale tickets dominate). This is *not* the RFM Monetary column yet.



In [ ]:
# Task 3.1 / 3.2 — country revenue + a quick spend histogram



## 4. Build the RFM table

`snapshot_date` is one day after the last invoice so Recency is a positive integer (the most recent buyer gets 1, never 0).

**Task 4** Group by `CustomerID`:

- Recency = `(snapshot - last InvoiceDate).days`
- Frequency = `nunique(InvoiceNo)`
- Monetary = `sum(TotalSum)`

Store a DataFrame named `rfm` with columns `CustomerID, Recency, Frequency, Monetary`. Print `describe()` and the raw skew.



In [ ]:
# Task 4 — snapshot + named aggregation → rfm
# snapshot = sales["InvoiceDate"].max() + pd.Timedelta(days=1)
# rfm = sales.groupby("CustomerID").agg(
#     Recency=("InvoiceDate", lambda x: (snapshot - x.max()).days),
#     Frequency=("InvoiceNo", "nunique"),
#     Monetary=("TotalSum", "sum"),
# ).reset_index()



## 5. Log1p then scale

Monetary skew on this extract is ~21. Frequency ~13. Recency is milder (~1.1) but still right-tailed. `np.log1p` compresses the whale tail; `StandardScaler` then puts days, invoices and pounds on the same footing.

**Task 5** Build `rfm_log` from the three columns via `np.log1p`. Fit a `StandardScaler` and store `rfm_scaled`. Print log-skew and the scaled column means / stds (should be ~0 / ~1). Keep original `rfm` untouched.



In [ ]:
# Task 5 — rfm_log and rfm_scaled
# rfm_log = rfm[["Recency", "Frequency", "Monetary"]].apply(np.log1p)
# scaler = StandardScaler()
# rfm_scaled = scaler.fit_transform(rfm_log)



## 6. Elbow and silhouette

Inertia always falls with \(k\). The useful question is where the drop flattens, cross-checked with silhouette.

**Task 6** For \(k = 1\ldots10\), fit `KMeans(n_clusters=k, random_state=42, n_init=10)` on `rfm_scaled`, store inertia. From \(k=2\) also store silhouette. Plot both. Mark a candidate \(k\).



In [ ]:
# Task 6 — elbow + silhouette lists and a two-axis plot



## 7. Fit K-Means and attach labels

The elbow on this extract is compatible with 3 or 4. Silhouette edges 4 over 3 (~0.325 vs 0.320). We take **k = 4** so the campaign team gets a Champions bin separate from recent-but-small buyers.

**Task 7** Fit `KMeans(n_clusters=4, random_state=42, n_init=10)` on `rfm_scaled`. Write the labels into **`rfm["Cluster"]`** — not into the log frame or the NumPy array.



In [ ]:
# Task 7 — k_optimal = 4; rfm["Cluster"] = ...



## 8. Profile in original units and name the bins

A centroid lives in log-scaled space. The email you send a store manager does not. Use **medians** — Monetary is still heavy-tailed in pounds.

**Task 8.1** `groupby("Cluster")` mean / median / count for Recency, Frequency, Monetary.

**Task 8.2** Customer share vs revenue share by cluster. Which bin is the whale group?

**Task 8.3** Give each integer a business name (Champions / Recent occasional / Slipping mid-value / Lost). Write `rfm["Segment"]`.



In [ ]:
# Task 8 — cluster_summary, value concentration, Segment names



## 9. PCA is a slide, not the model

Three scaled columns → two principal components. Colour by Cluster.

**Task 9** `PCA(n_components=2)` on `rfm_scaled`. Build a frame with `PC1`, `PC2`, `Cluster`. Seaborn or matplotlib scatter. Print explained-variance ratios and the 3×2 loading matrix (rows = R, F, M).



In [ ]:
# Task 9 — PCA scatter + loadings



## 10. Alternate code that reaches the same idea

A. `fit_predict` instead of `fit` + `.labels_`.
B. Scale with `RobustScaler` (or `MinMaxScaler`) after the same `log1p`. Does k=4 membership move?
C. Quintile RFM scores (`pd.qcut`, Recency inverted) instead of K-Means — a 555 / 111 style card.
D. `AgglomerativeClustering(n_clusters=4)` or `MiniBatchKMeans` on the same `rfm_scaled`.
E. NumPy SVD instead of `sklearn.decomposition.PCA`.



In [ ]:
# Task 10 — pick at least two alternates and compare cluster sizes / medians



## 11. More practice

**P1.** Rebuild RFM on the United Kingdom only. Refit k=4. Do the same four names still fit?

**P2.** Restrict the sales window to the last 180 days before the snapshot. Recency is compressed; Frequency and Monetary shrink. What happens to the Champion share of revenue?

**P3.** Score one new customer who is not in the table: Recency=7, Frequency=6, Monetary=400. Transform with the **training** log1p + scaler (`scaler.transform`), then `kmeans.predict`.

**P4.** Drop Frequency and cluster on Recency + Monetary only. Which segment collapses?



In [ ]:
# Task 11 — P1 UK-only / P2 180-day window / P3 one new customer / P4 drop Frequency



## 12. Simulation — turn the knobs

Modify a few values and watch inertia / silhouette / Champion revenue-share move.

Knobs:

- `k` — 2 to 8
- `n` — subsample of customers (500, 1500, all)
- `noise` — Gaussian σ added to `rfm_scaled`
- `n_init` — 1 vs 10 (bad init vs stable)
- `tenure_days` — rebuild Monetary/Frequency on a trailing window

Use `CustSeg.simulate` or write a small `run_once`. Plot inertia and silhouette against k at two noise levels.



In [ ]:
# Task 12 — knobs
K = 4
N = None          # None = all customers
NOISE = 0.0
N_INIT = 10
SEED = 42

# result = cs.simulate(rfm_scaled, k=K, n=N, noise=NOISE, n_init=N_INIT, seed=SEED)
# print(result)

# then a small grid: k in 2..8 × noise in {0, 0.15, 0.3}



## Audience rewrite (Jočys checklist + McMurrey types)

| Audience | What they need | One sentence they should hear |
|----------|----------------|-------------------------------|
| Expert (CRM scientist) | inertia, silhouette, loadings, log+scale order | k=4 is compatible with the elbow; PC1 (71%) is value, PC2 (21%) is recency; name bins from medians. |
| Technician (campaign ops) | a four-row treatment table and the query keys | Champions = cluster 2 on this seed; suppress win-back mail to cluster 1 until a human reviews the offer. |
| Executive (CMO / retail) | concentration, not eigenvalues | 17% of customers produce 64% of observed revenue; the slipping mid-value bin is the cheapest win-back. |
| Nonspecialist | a store-floor analogy | We sorted shoppers by *how lately, how often, how much* — not by postcode or first name. |

Literacy: bars and a 2-D scatter, not a 3-D spinning cube. Subject knowledge: do not expand RFM. Time span: one slide of concentration + one treatment table.



## What this model can and cannot do

**Can**
- Collapse invoices to one RFM row per customer and group similar rows.
- Show value concentration (Champion share of revenue vs headcount).
- Give campaign ops four named bins and a reproducible seed.

**Cannot**
- Predict the next SKU, the next date, or the next pound.
- Replace a loyalty tier, a credit limit, or a GDPR lawful-basis review.
- Travel unchanged to another banner, currency, or year without refitting the scaler.
- Treat `Cluster == 2` as a permanent VIP badge — ids shuffle if you change k or the seed.

**Top uses:** CRM design, win-back vs nurture split, wholesale-vs-retail separation, board concentration slides.
**Anti-uses:** automated discounting without a human, scoring a customer with a different scaler, clustering on raw unlogged Monetary.



## Next steps

- Add country or channel as a feature (scale the dummy; it will fight Recency).
- Try k=3 if ops will only fund three treatments.
- Cap Monetary at the 99th percentile before log1p and re-check Champion size.
- Read `CustSeg_Strategy_Guide.docx` before you clone this onto another ledger.

